<a href="https://colab.research.google.com/github/Somtochukwu438/quizgen-llama3/blob/main/notebooks/03_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers peft bitsandbytes accelerate datasets evaluate rouge_score bert_score
from google.colab import userdata, drive
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, sys
REPO = "/content/quizgen-llama3"
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Somtochukwu438/quizgen-llama3.git {REPO}
sys.path.append(REPO)
from src.prompts import build_messages
from src.parse import parse_output

OUT = "/content/drive/MyDrive/quizgen"
!ls -lh {REPO}/adapters

Cloning into '/content/quizgen-llama3'...
remote: Enumerating objects: 45, done.
remote: Total 45 (delta 0), reused 0 (delta 0), pack-reused 45 (from 1)
Receiving objects: 100% (45/45), 42.44 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (17/17), done.
total 47M
-rw-r--r-- 1 root root 1.2K Nov  2 19:58 adapter_config.json
-rw-r--r-- 1 root root  47M Nov  2 19:58 adapter_model.safetensors


In [4]:
from datasets import load_dataset
sciq = load_dataset("allenai/sciq")
test = sciq["test"].filter(lambda x: x["support"].strip() != "")
eval_set = test.shuffle(seed=42).select(range(300))
print(len(eval_set), "evaluation passages")

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

300 evaluation passages


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, f"{REPO}/adapters")
model.eval()
print("Fine-tuned model loaded")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Fine-tuned model loaded


In [7]:
def generate(passages, batch_size=8, max_new_tokens=200):
    outs = []
    for i in range(0, len(passages), batch_size):
        batch = passages[i : i + batch_size]
        prompts = [
            tok.apply_chat_template(build_messages(p), tokenize=False, add_generation_prompt=True)
            for p in batch
        ]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{len(outs)}/{len(passages)}")
    return outs

In [8]:
import json
from collections import Counter

raw = generate(eval_set["support"])

records = []
for ex, out in zip(eval_set, raw):
    parsed, err = parse_output(out)
    records.append({
        "passage": ex["support"],
        "ref_question": ex["question"],
        "ref_answer": ex["correct_answer"],
        "ref_distractors": [ex["distractor1"], ex["distractor2"], ex["distractor3"]],
        "raw": out,
        "parsed": parsed,
        "error": err,
    })

with open(f"{OUT}/finetuned_outputs.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

n = len(records)
valid = sum(r["parsed"] is not None for r in records)
strict = sum(r["parsed"] is not None and r["parsed"]["strict"] for r in records)
print(f"Valid:  {valid}/{n} ({valid/n:.1%})")
print(f"Strict: {strict}/{n} ({strict/n:.1%})")
print("Errors:", Counter(r["error"] for r in records if r["error"]))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


8/300
16/300
24/300
32/300
40/300
48/300
56/300
64/300
72/300
80/300
88/300
96/300
104/300
112/300
120/300
128/300
136/300
144/300
152/300
160/300
168/300
176/300
184/300
192/300
200/300
208/300
216/300
224/300
232/300
240/300
248/300
256/300
264/300
272/300
280/300
288/300
296/300
300/300
Valid:  238/300 (79.3%)
Strict: 238/300 (79.3%)
Errors: Counter({'answer_in_distractors': 60, 'no_json': 2})


In [9]:
bad = [r for r in records if r["error"] == "answer_in_distractors"]
for r in bad[:8]:
    print(r["raw"][:300].replace("\n", " "))
    print("-" * 70)

{"question": "What is the protein in red blood cells which transports oxygen around the body?", "answer": "hemoglobin", "distractors": ["hematocrit", "hemoglobin", "hemoglobin"]}
----------------------------------------------------------------------
{"question": "What do we call the first step in extracting metals from the earth?", "answer": "mining", "distractors": ["excavation", "mining", "excavation"]}
----------------------------------------------------------------------
{"question": "What is the first step in the process of cell division?", "answer": "mitosis", "distractors": ["meiosis", "mitosis", "prophase"]}
----------------------------------------------------------------------
{"question": "What is water more dense in?", "answer": "liquid state", "distractors": ["solid state", "gas state", "liquid state"]}
----------------------------------------------------------------------
{"question": "What are the products of photosynthesis needed for?", "answer": "cellular respiration", 

In [10]:
norm = lambda s: s.strip().lower()
train_check = (sciq["train"].filter(lambda x: x["support"].strip() != "")
               .shuffle(seed=42).select(range(2000)))
dup = sum(norm(ex["correct_answer"]) in {norm(ex[f"distractor{i}"]) for i in (1, 2, 3)}
          for ex in train_check)
print(dup, "of 2000 training examples have the answer among the distractors")

Filter:   0%|          | 0/11679 [00:00<?, ? examples/s]

2 of 2000 training examples have the answer among the distractors


In [11]:
for r in records[:5]:
    print("REFERENCE:", r["ref_question"], "|", r["ref_answer"])
    if r["parsed"]:
        p = r["parsed"]
        print("FINE-TUNED:", p["question"], "|", p["answer"])
        print("DISTRACTORS:", p["distractors"])
    else:
        print("FAILED:", r["error"], "|", r["raw"][:200])
    print("=" * 70)

REFERENCE: By the year 2050, 25 percent of the population of the united states will be 60 years of age or older. the cdc estimates that 80 percent of those 60 years and older have one or more chronic disease associated with deficiencies of this? | immune system
FINE-TUNED: What is the term for the loss of immune function with age? | immunosenescence
DISTRACTORS: ['aging', 'inflammation', 'infection']
REFERENCE: Enzymes can catalyze up to several million reactions per what? | second
FINE-TUNED: What are extremely efficient in speeding up biochemical reactions? | enzymes
DISTRACTORS: ['proteins', 'ions', 'molecules']
REFERENCE: What are classified by climatic factors and types of primary producers? | terrestrial biomes
FINE-TUNED: What are classified by climatic factors and types of primary producers? | terrestrial biomes
DISTRACTORS: ['terrestrial ecosystems', 'terrestrial organisms', 'terrestrial plants']
REFERENCE: What is the protein in red blood cells which transports oxygen around 

In [13]:
!git -C {REPO} pull
import importlib, src.parse
importlib.reload(src.parse)
from src.parse import parse_output

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 682 bytes | 682.00 KiB/s, done.
From https://github.com/Somtochukwu438/quizgen-llama3
   e09b4a9..228ea0a  main       -> origin/main
Updating e09b4a9..228ea0a
Fast-forward
 src/parse.py | 16 +++++++++++-----
 1 file changed, 11 insertions(+), 5 deletions(-)


In [14]:
import json
import pandas as pd

def load(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

results = {
    "Baseline": load(f"{OUT}/baseline_outputs.jsonl"),
    "Fine-tuned": load(f"{OUT}/finetuned_outputs.jsonl"),
}
assert all(b["passage"] == f["passage"]
           for b, f in zip(results["Baseline"], results["Fine-tuned"])), "passage order mismatch"

for recs in results.values():
    for r in recs:
        r["parsed"], r["error"] = parse_output(r["raw"])

def summarize(recs):
    n = len(recs)
    ok = [r["parsed"] for r in recs if r["parsed"]]
    rep = sum(p["answer_repeated"] for p in ok)
    dup = sum(p["distractor_duplicates"] for p in ok)
    clean = sum(not p["answer_repeated"] and not p["distractor_duplicates"] for p in ok)
    return {
        "Format valid": f"{len(ok)}/{n} ({len(ok)/n:.1%})",
        "Answer repeated": f"{rep}/{n} ({rep/n:.1%})",
        "Duplicate distractors": f"{dup}/{n} ({dup/n:.1%})",
        "Fully clean": f"{clean}/{n} ({clean/n:.1%})",
    }

table = pd.DataFrame({name: summarize(recs) for name, recs in results.items()})
print(table)

                              Baseline       Fine-tuned
Format valid           292/300 (97.3%)  298/300 (99.3%)
Answer repeated           5/300 (1.7%)   60/300 (20.0%)
Duplicate distractors     0/300 (0.0%)   83/300 (27.7%)
Fully clean            287/300 (95.7%)  187/300 (62.3%)


In [15]:
import evaluate

both = [i for i in range(300)
        if results["Baseline"][i]["parsed"] and results["Fine-tuned"][i]["parsed"]]
print(len(both), "passages where both models produced valid JSON")

refs = [results["Baseline"][i]["ref_question"] for i in both]
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

sim = {}
for name, recs in results.items():
    preds = [recs[i]["parsed"]["question"] for i in both]
    r = rouge.compute(predictions=preds, references=refs, use_stemmer=True)["rougeL"]
    b = bertscore.compute(predictions=preds, references=refs, lang="en",
                          rescale_with_baseline=True)
    sim[name] = {"ROUGE-L": round(r, 3),
                 "BERTScore F1": round(sum(b["f1"]) / len(b["f1"]), 3)}
print(pd.DataFrame(sim))

290 passages where both models produced valid JSON


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


              Baseline  Fine-tuned
ROUGE-L          0.294       0.452
BERTScore F1     0.387       0.482


In [18]:
import torch, re, string, collections
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_name = "deepset/roberta-base-squad2"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name).to("cuda").eval()

def qa_answer(questions, passages, batch_size=16, max_answer_len=30):
    """Extractive QA: return the passage span that answers each question, or "" if none."""
    answers = []
    for i in range(0, len(questions), batch_size):
        qs, ps = questions[i:i + batch_size], passages[i:i + batch_size]
        enc = qa_tok(qs, ps, truncation="only_second", max_length=512, padding=True,
                     return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")
        with torch.no_grad():
            out = qa_model(**{k: v.to("cuda") for k, v in enc.items()})
        for j in range(len(qs)):
            s, e = out.start_logits[j].cpu(), out.end_logits[j].cpu()
            ctx = torch.tensor([k for k, sid in enumerate(enc.sequence_ids(j)) if sid == 1])
            k = min(20, len(ctx))
            starts = ctx[s[ctx].topk(k).indices].tolist()
            ends = ctx[e[ctx].topk(k).indices].tolist()
            best, span = float("-inf"), None
            for a in starts:
                for b in ends:
                    if a <= b < a + max_answer_len and s[a] + e[b] > best:
                        best, span = (s[a] + e[b]).item(), (a, b)
            null_score = (s[0] + e[0]).item()
            if span is None or null_score > best:
                answers.append("")
            else:
                answers.append(ps[j][offsets[j][span[0]][0]:offsets[j][span[1]][1]])
    return answers

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def token_f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    same = sum((collections.Counter(p) & collections.Counter(g)).values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)

def answerable(questions, answers, passages, threshold=0.5):
    preds = qa_answer(questions, passages)
    return [token_f1(p, a) >= threshold for p, a in zip(preds, answers)]

passages = [results["Baseline"][i]["passage"] for i in both]
ref_answers = [results["Baseline"][i]["ref_answer"] for i in both]

hits = {"SciQ reference (ceiling)": answerable(refs, ref_answers, passages)}
for name, recs in results.items():
    qs = [recs[i]["parsed"]["question"] for i in both]
    golds = [recs[i]["parsed"]["answer"] for i in both]
    hits[name] = answerable(qs, golds, passages)

print(pd.Series({k: round(sum(v) / len(v), 3) for k, v in hits.items()}, name="Answerable"))

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SciQ reference (ceiling)    0.897
Baseline                    0.648
Fine-tuned                  0.824
Name: Answerable, dtype: float64


In [19]:
from scipy.stats import binomtest

b, f = hits["Baseline"], hits["Fine-tuned"]
only_ft = sum(fi and not bi for bi, fi in zip(b, f))
only_base = sum(bi and not fi for bi, fi in zip(b, f))
p = binomtest(only_ft, only_ft + only_base, 0.5).pvalue
print(f"Answerable only for fine-tuned: {only_ft} | only for baseline: {only_base} | p = {p:.2g}")

for name, recs in results.items():
    lens = [len(recs[i]["parsed"]["answer"].split()) for i in both]
    print(f"{name}: mean answer length = {sum(lens) / len(lens):.2f} words")

Answerable only for fine-tuned: 75 | only for baseline: 24 | p = 2.8e-07
Baseline: mean answer length = 4.40 words
Fine-tuned: mean answer length = 1.61 words


In [20]:
rows = {}
for name in results:
    rows[name] = {**summarize(results[name]), **sim[name],
                  "Answerable": round(sum(hits[name]) / len(hits[name]), 3)}
rows["SciQ reference"] = {"Answerable": round(sum(hits["SciQ reference (ceiling)"]) / len(both), 3)}
final = pd.DataFrame(rows)
final.to_csv(f"{OUT}/results_table.csv")
print(final)

                              Baseline       Fine-tuned  SciQ reference
Format valid           292/300 (97.3%)  298/300 (99.3%)             NaN
Answer repeated           5/300 (1.7%)   60/300 (20.0%)             NaN
Duplicate distractors     0/300 (0.0%)   83/300 (27.7%)             NaN
Fully clean            287/300 (95.7%)  187/300 (62.3%)             NaN
ROUGE-L                          0.294            0.452             NaN
BERTScore F1                     0.387            0.482             NaN
Answerable                       0.648            0.824           0.897


In [21]:
short = [k for k, i in enumerate(both)
         if len(results["Baseline"][i]["parsed"]["answer"].split()) <= 2
         and len(results["Fine-tuned"][i]["parsed"]["answer"].split()) <= 2]
print(len(short), "passages where both answers are 2 words or fewer")
for name in ["Baseline", "Fine-tuned"]:
    rate = sum(hits[name][k] for k in short) / len(short)
    print(f"{name}: answerable = {rate:.3f}")

109 passages where both answers are 2 words or fewer
Baseline: answerable = 0.670
Fine-tuned: answerable = 0.798


In [22]:
from collections import Counter

def generate_sampled(passages, batch_size=8, max_new_tokens=200):
    outs = []
    for i in range(0, len(passages), batch_size):
        batch = passages[i : i + batch_size]
        prompts = [tok.apply_chat_template(build_messages(p), tokenize=False, add_generation_prompt=True)
                   for p in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                                 temperature=0.7, top_p=0.9, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def needs_retry(parsed):
    return parsed is None or parsed["answer_repeated"] or parsed["distractor_duplicates"]

def regenerate(recs, use_adapter, max_tries=3):
    recs = [dict(r) for r in recs]
    for attempt in range(1, max_tries + 1):
        idx = [i for i, r in enumerate(recs) if needs_retry(r["parsed"])]
        if not idx:
            break
        ps = [recs[i]["passage"] for i in idx]
        if use_adapter:
            outs = generate_sampled(ps)
        else:
            with model.disable_adapter():
                outs = generate_sampled(ps)
        for i, out in zip(idx, outs):
            recs[i]["raw"] = out
            recs[i]["parsed"], recs[i]["error"] = parse_output(out)
        print(f"  attempt {attempt}: retried {len(idx)} outputs")
    return recs

In [23]:
torch.manual_seed(42)
regen = {}
for name, use_adapter in [("Baseline", False), ("Fine-tuned", True)]:
    print(name)
    regen[name] = regenerate(results[name], use_adapter)
    with open(f"{OUT}/{name.lower().replace('-', '')}_regen_outputs.jsonl", "w") as f:
        for r in regen[name]:
            f.write(json.dumps(r) + "\n")

print(pd.DataFrame({
    "Baseline (raw)": summarize(results["Baseline"]),
    "Baseline + regen": summarize(regen["Baseline"]),
    "Fine-tuned (raw)": summarize(results["Fine-tuned"]),
    "Fine-tuned + regen": summarize(regen["Fine-tuned"]),
}))

Baseline
  attempt 1: retried 13 outputs
  attempt 2: retried 2 outputs
Fine-tuned
  attempt 1: retried 113 outputs
  attempt 2: retried 25 outputs
  attempt 3: retried 6 outputs
                        Baseline (raw)  Baseline + regen Fine-tuned (raw)  \
Format valid           292/300 (97.3%)  300/300 (100.0%)  298/300 (99.3%)   
Answer repeated           5/300 (1.7%)      0/300 (0.0%)   60/300 (20.0%)   
Duplicate distractors     0/300 (0.0%)      0/300 (0.0%)   83/300 (27.7%)   
Fully clean            287/300 (95.7%)  300/300 (100.0%)  187/300 (62.3%)   

                      Fine-tuned + regen  
Format valid            300/300 (100.0%)  
Answer repeated             2/300 (0.7%)  
Duplicate distractors       1/300 (0.3%)  
Fully clean              298/300 (99.3%)  


In [24]:
refs_all = [r["ref_question"] for r in regen["Baseline"]]
passages_all = [r["passage"] for r in regen["Baseline"]]

regen_rows = {}
for name, recs in regen.items():
    preds = [r["parsed"]["question"] if r["parsed"] else "" for r in recs]
    golds = [r["parsed"]["answer"] if r["parsed"] else "" for r in recs]
    rl = rouge.compute(predictions=preds, references=refs_all, use_stemmer=True)["rougeL"]
    bs = bertscore.compute(predictions=preds, references=refs_all, lang="en",
                           rescale_with_baseline=True)
    h = answerable(preds, golds, passages_all)
    regen_rows[f"{name} + regen"] = {
        **summarize(recs),
        "ROUGE-L": round(rl, 3),
        "BERTScore F1": round(sum(bs["f1"]) / len(bs["f1"]), 3),
        "Answerable": round(sum(h) / len(h), 3),
    }

regen_table = pd.DataFrame(regen_rows)
regen_table.to_csv(f"{OUT}/results_table_regen.csv")
print(regen_table)

                       Baseline + regen Fine-tuned + regen
Format valid           300/300 (100.0%)   300/300 (100.0%)
Answer repeated            0/300 (0.0%)       2/300 (0.7%)
Duplicate distractors      0/300 (0.0%)       1/300 (0.3%)
Fully clean            300/300 (100.0%)    298/300 (99.3%)
ROUGE-L                           0.292              0.443
BERTScore F1                      0.385              0.474
Answerable                         0.65               0.82
